### Seiten zu INV/UNINV

In [7]:
import re
import pandas as pd
from openpyxl import load_workbook

HEALTHY_XLSX = r"K:\Team\Böhmer_Michael\PAPER\Maestroni_Validierungsdaten.xlsx"
HEALTHY_SHEET = "Healthy"

INV_XLSX = r"K:\Team\Böhmer_Michael\PAPER\Validierungsdaten_INV_UNINV\Basistabelle_Maestroni_ML.xlsx"
INV_SHEET_READ = 0   # so liest du per Index …
# … aber zum Schreiben brauchen wir den NAMEN:
wb = load_workbook(INV_XLSX, read_only=True)
INV_SHEET_WRITE = wb.sheetnames[INV_SHEET_READ]   # z.B. "Tabelle1" o.ä.

def norm(s):
    s = str(s).strip().replace(" ", "_")
    return re.sub(r"__+", "_", s)

def split_lr(c):
    c = norm(c)
    m = re.search(r"_(R|L)$", c, flags=re.I)
    if m: return c[:-(len(m.group(0)))], m.group(1).upper()
    m = re.search(r"_(Right|Left)$", c, flags=re.I)
    if m: return c[:-(len(m.group(0)))], ('R' if m.group(1).lower()=='right' else 'L')
    return c, None

def split_inv(c):
    c = norm(c)
    m = re.search(r"_(INV|UNINV)$", c, flags=re.I)
    if m: return c[:-(len(m.group(0)))], m.group(1).upper()
    return c, None

# --- Einlesen ---
dfH = pd.read_excel(HEALTHY_XLSX, sheet_name=HEALTHY_SHEET, engine="openpyxl")
dfI = pd.read_excel(INV_XLSX, sheet_name=INV_SHEET_READ, engine="openpyxl")

# Normalisieren
dfH.columns = [norm(c) for c in dfH.columns]
dfI.columns = [norm(c) for c in dfI.columns]

# LR-Paare (Healthy) und INV/UNINV-Paare (INV)
H_pairs, I_pairs = {}, {}
for c in dfH.columns:
    b, lr = split_lr(c)
    if lr: H_pairs.setdefault(b, {})[lr] = c
for c in dfI.columns:
    b, tag = split_inv(c)
    if tag: I_pairs.setdefault(b, {})[tag] = c

# Nicht-seitige Spalten 1:1 kopieren (falls vorhanden)
for col in sorted(set(dfH.columns) & set(dfI.columns)):
    dfI[col] = dfH[col]

# Dominanz
DOM_COL = 'Limb_Dominance'  # wie von dir beschrieben
dom = dfH[DOM_COL].astype(str).str.strip().str.lower()

# Seitige Metriken mappen (dominant → INV, nicht-dominant → UNINV)
common = sorted(set(H_pairs) & set(I_pairs))
rows = min(len(dfH), len(dfI))

for b in common:
    srcL, srcR = H_pairs[b].get('L'), H_pairs[b].get('R')
    dstINV, dstUNINV = I_pairs[b].get('INV'), I_pairs[b].get('UNINV')
    if not (srcL and srcR and dstINV and dstUNINV):
        continue
    for i in range(rows):
        d = dom.iat[i]
        if d in ('left', 'l'):
            dfI.at[i, dstINV]   = dfH.at[i, srcL]
            dfI.at[i, dstUNINV] = dfH.at[i, srcR]
        elif d in ('right', 'r'):
            dfI.at[i, dstINV]   = dfH.at[i, srcR]
            dfI.at[i, dstUNINV] = dfH.at[i, srcL]

# --- Zurückschreiben: sheet_name MUSS String sein ---
with pd.ExcelWriter(INV_XLSX, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    dfI.to_excel(xw, sheet_name=INV_SHEET_WRITE, index=False)


### Seiten zu INV-UNINV für Maestroni-Daten

In [2]:
import re
import pandas as pd
from openpyxl import load_workbook
import warnings

# Optional: Pandas FutureWarnings unterdrücken (saubere Konsole)
warnings.filterwarnings("ignore", category=FutureWarning)

PATH = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Feature_Selektion_Maestroni_Daten\Basistabelle_ML_Maestroni_gewichtsnormalisiert_inv.xlsx"
SHEET_SRC = "umsortieren"
SHEET_DST = "neu"
DOM_COL = "Dominanz"  # erwartet in "umsortieren"

def norm(s: str) -> str:
    s = str(s).strip().replace(" ", "_")
    return re.sub(r"__+", "_", s)

def split_lr(col: str):
    """
    Zerlegt Spalten mit ..._L / ..._R am Ende.
    Rückgabe: (basisname, 'L'/'R' oder None)
    """
    c = norm(col)
    m = re.search(r"_(L|R)$", c, flags=re.I)
    if m:
        return c[:-(len(m.group(0)))], m.group(1).upper()
    m = re.search(r"_(Left|Right)$", c, flags=re.I)
    if m:
        return c[:-(len(m.group(0)))], ('L' if m.group(1).lower() == 'left' else 'R')
    return c, None

def extract_inv_base(col: str):
    """
    Für Zielnamen aus Sheet 'neu': INV_Base oder UNINV_Base -> ('INV'/'UNINV', 'Base')
    Sonst (nicht-seitig) -> (None, original)
    """
    c = norm(col)
    if c.upper().startswith("INV_"):
        return "INV", c[4:]
    if c.upper().startswith("UNINV_"):
        return "UNINV", c[6:]
    return None, c

# --- Einlesen ---
df_src = pd.read_excel(PATH, sheet_name=SHEET_SRC, engine="openpyxl")
df_dst_template = pd.read_excel(PATH, sheet_name=SHEET_DST, engine="openpyxl")

# Spaltennamen normalisieren (vereinheitlichen für Matching)
src_cols = [norm(c) for c in df_src.columns]
dst_cols = [norm(c) for c in df_dst_template.columns]
df_src.columns = src_cols
df_dst_template.columns = dst_cols

# Prüfe Dominanzspalte
if DOM_COL not in df_src.columns:
    raise ValueError(f"Dominanz-Spalte '{DOM_COL}' fehlt in Sheet '{SHEET_SRC}'.")

dom = df_src[DOM_COL].astype(str).str.strip().str.lower()  # 'left'/'l' oder 'right'/'r'
nrows = len(df_src)

# Mappe L/R-Quellspalten: basis -> {'L': colname, 'R': colname}
lr_map = {}
for c in df_src.columns:
    base, side = split_lr(c)
    if side in ('L', 'R'):
        lr_map.setdefault(base, {})[side] = c

# Ergebnis-DF mit den Zielspalten aus 'neu'
out = pd.DataFrame(index=range(nrows), columns=df_dst_template.columns, dtype="object")

# 1) Nicht-seitige Zielspalten: wenn identischer Name im src existiert -> 1:1 kopieren
for col in out.columns:
    tag, base = extract_inv_base(col)
    if tag is None:
        # nicht-seitig
        if base in df_src.columns:
            out[col] = df_src[base]
        else:
            # falls der Zielname bereits genau so im src existiert (ohne Normierung)
            out[col] = pd.NA

# 2) Seitige Zielspalten (INV_/UNINV_): per Dominanz aus _L/_R füllen
for col in out.columns:
    tag, base = extract_inv_base(col)
    if tag is None:
        continue  # schon in (1) behandelt

    # Benötigt: zu diesem base existieren _L und _R im src
    pair = lr_map.get(base, {})
    srcL, srcR = pair.get('L'), pair.get('R')
    if not (srcL and srcR):
        # Kein passendes L/R-Paar -> leer lassen
        continue

    # Zeilenweise zuweisen
    # INV = dominantes Bein; UNINV = nicht-dominantes
    vals = []
    for i in range(nrows):
        d = dom.iat[i]
        if d in ("left", "l"):
            inv_val = df_src.at[i, srcL]
            uninv_val = df_src.at[i, srcR]
        elif d in ("right", "r"):
            inv_val = df_src.at[i, srcR]
            uninv_val = df_src.at[i, srcL]
        else:
            inv_val = pd.NA
            uninv_val = pd.NA

        vals.append(inv_val if tag == "INV" else uninv_val)

    out[col] = vals

# Optional: gleiche Reihenfolge wie Vorlage beibehalten
out = out[df_dst_template.columns]

# Zurückschreiben (ersetzt Sheet 'neu')
with pd.ExcelWriter(PATH, engine="openpyxl", mode="a", if_sheet_exists="replace") as xw:
    out.to_excel(xw, sheet_name=SHEET_DST, index=False)

print(f"Umsortierung abgeschlossen. Sheet '{SHEET_DST}' wurde aktualisiert.")


Umsortierung abgeschlossen. Sheet 'neu' wurde aktualisiert.


### Sortierung nach Header

In [1]:
# -*- coding: utf-8 -*-
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.utils import get_column_letter

FILE_PATH = r"K:\Team\Böhmer_Michael\PAPER\Maestroni_Validierungsdaten.xlsx"
SHEET_NAME = "ACLR"

# Umsortierbereich C–U, Block 3–35 (inkl. Ist-Header in Zeile 3)
COL_START = 3   # C
COL_END   = 21  # U
ROW_DATA_START = 3   # <-- Block beginnt in Zeile 3 (Ist-Header)
ROW_DATA_END   = 35

# Zielreihenfolge kommt aus Zeile 1
ROW_TARGET_ORDER = 1
NEW_SHEET_NAME   = "ACLR_reordered"

def normalize_label(s: Optional[str]) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    if not s:
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    for ch in ["\n", "\r", "\t", "_", "-", "–", "—", "/", "\\", "|"]:
        s = s.replace(ch, " ")
    return " ".join(s.split()).lower()

def read_cell(ws: Worksheet, row: int, col: int):
    return ws.cell(row=row, column=col).value

def write_cell(ws: Worksheet, row: int, col: int, value):
    ws.cell(row=row, column=col, value=value)

def get_target_order(ws: Worksheet) -> Tuple[List[str], List[str]]:
    raw, norm = [], []
    for c in range(COL_START, COL_END + 1):
        v = read_cell(ws, ROW_TARGET_ORDER, c)
        raw.append(v if v is not None else "")
        norm.append(normalize_label(v))
    return raw, norm

def get_source_labels_from_row3(ws: Worksheet) -> Tuple[List[str], List[str]]:
    """Liest die Ist-Header aus Zeile 3 des Umsortierbereichs (C3..U3)."""
    raw, norm = [], []
    for c in range(COL_START, COL_END + 1):
        v = read_cell(ws, ROW_DATA_START, c)  # Zeile 3
        raw.append(v if v is not None else "")
        norm.append(normalize_label(v))
    return raw, norm

def build_mapping(src_norm: List[str], tgt_norm: List[str]):
    label_to_tgt = {}
    for i, lab in enumerate(tgt_norm):
        label_to_tgt.setdefault(lab, []).append(i)

    dup_targets = {lab: idxs for lab, idxs in label_to_tgt.items() if len(idxs) > 1 and lab != ""}

    colmap: Dict[int, int] = {}
    used_tgt = set()
    for s_idx, slab in enumerate(src_norm):
        if slab == "":
            continue
        for ti in label_to_tgt.get(slab, []):
            if ti not in used_tgt:
                colmap[s_idx] = ti
                used_tgt.add(ti)
                break

    missing_targets = [ti for ti in range(len(tgt_norm)) if ti not in used_tgt]
    extras_sources  = [si for si in range(len(src_norm)) if si not in colmap]
    return colmap, missing_targets, extras_sources, dup_targets

def buffer_blocks(ws: Worksheet, colmap: Dict[int, int]) -> Dict[int, List]:
    """Puffert den gesamten Block Zeile 3–35 der jeweils zugeordneten Quellspalte."""
    buf: Dict[int, List] = {}
    for s_idx, t_idx in colmap.items():
        abs_col = COL_START + s_idx
        data = [read_cell(ws, r, abs_col) for r in range(ROW_DATA_START, ROW_DATA_END + 1)]
        buf[t_idx] = data
    return buf

def create_or_replace_sheet(wb, name: str) -> Worksheet:
    if name in wb.sheetnames:
        wb.remove(wb[name])
    return wb.create_sheet(title=name)

def main():
    xlsx = Path(FILE_PATH)
    if not xlsx.exists():
        raise FileNotFoundError(f"File not found: {xlsx}")
    wb = load_workbook(xlsx)
    if SHEET_NAME not in wb.sheetnames:
        raise ValueError(f"Sheet '{SHEET_NAME}' not found.")
    ws_src: Worksheet = wb[SHEET_NAME]

    # Ziel (Zeile 1) und Ist (Zeile 3) lesen
    tgt_raw, tgt_norm = get_target_order(ws_src)
    src_raw, src_norm = get_source_labels_from_row3(ws_src)

    colmap, missing_targets, extras_sources, dup_targets = build_mapping(src_norm, tgt_norm)

    # Phase 1: Puffer
    buffer = buffer_blocks(ws_src, colmap)

    # Phase 2: Schreiben in neues Blatt
    ws_out = create_or_replace_sheet(wb, NEW_SHEET_NAME)

    # (Optional) Linke Spalten A..B kopieren (Zeilen 1..ROW_DATA_END)
    for c in range(1, COL_START):
        for r in range(1, ROW_DATA_END + 1):
            write_cell(ws_out, r, c, read_cell(ws_src, r, c))

    # Zeile 1 (Ziel-Header) setzen
    for i, raw_label in enumerate(tgt_raw):
        write_cell(ws_out, ROW_TARGET_ORDER, COL_START + i, raw_label)

    # Zeile 2 bewusst leer lassen (keine separaten Header schreiben)
    for i in range(COL_END - COL_START + 1):
        write_cell(ws_out, 2, COL_START + i, None)

    # Block Zeilen 3–35 gemäß Mapping schreiben
    for t_idx in range(COL_END - COL_START + 1):
        abs_col_out = COL_START + t_idx
        if t_idx in buffer:
            data = buffer[t_idx]
            for off, val in enumerate(data):
                write_cell(ws_out, ROW_DATA_START + off, abs_col_out, val)
        else:
            # fehlende Zielspalte leer lassen
            for r in range(ROW_DATA_START, ROW_DATA_END + 1):
                write_cell(ws_out, r, abs_col_out, None)

    # Extras rechts anhängen (optional)
    if extras_sources:
        out_c = COL_END + 1
        for s_idx in extras_sources:
            abs_col_src = COL_START + s_idx
            # kopiere auch Zeilen 1..35 für Transparenz
            for r in range(1, ROW_DATA_END + 1):
                write_cell(ws_out, r, out_c, read_cell(ws_src, r, abs_col_src))
            out_c += 1

    # Report
    print("=== Reorder Summary ===")
    print(f"Source sheet  : {SHEET_NAME}")
    print(f"Output sheet  : {NEW_SHEET_NAME}")
    print(f"Block moved   : {get_column_letter(COL_START)}{ROW_DATA_START}–{get_column_letter(COL_END)}{ROW_DATA_END}")
    print(f"Mapped columns: {len(colmap)} / {COL_END - COL_START + 1}")
    if dup_targets:
        print("Duplicate target labels (normalized):")
        for lab, idxs in dup_targets.items():
            cols = [get_column_letter(COL_START + i) for i in idxs]
            print(f"  '{lab}' at targets {cols}")
    if missing_targets:
        cols = [get_column_letter(COL_START + i) for i in missing_targets]
        names = [tgt_raw[i] for i in missing_targets]
        print(f"Missing targets: {list(zip(cols, names))}")
    if extras_sources:
        cols = [get_column_letter(COL_START + i) for i in extras_sources]
        names = [src_raw[i] for i in extras_sources]
        print(f"Extra sources : {list(zip(cols, names))}")

    wb.save(xlsx)
    print(f"Saved: {xlsx}")
    print("Done.")

if __name__ == "__main__":
    main()


=== Reorder Summary ===
Source sheet  : ACLR
Output sheet  : ACLR_reordered
Block moved   : C3–U35
Mapped columns: 19 / 19
Saved: K:\Team\Böhmer_Michael\PAPER\Maestroni_Validierungsdaten.xlsx
Done.


### Matching Variablen Maestroni und unsere Daten (gelb färben)

In [3]:
import os
import shutil
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

# --- Pfade ---
MAESTRONI_PATH = r"K:\Team\Böhmer_Michael\PAPER\Validierungsdaten_INV_UNINV\Basistabelle_Maestroni_ML.xlsx"
UNSERE_PATH    = r"K:\Team\Böhmer_Michael\PAPER\Validierungsdaten_INV_UNINV\Basistabelle_ML_reduziert_highlighted.xlsx"

MAESTRONI_SHEET = None
UNSERE_SHEET    = None

YELLOW = PatternFill(fill_type="solid", start_color="FFFF00", end_color="FFFF00")

def norm(s):
    return str(s).strip().casefold() if s is not None else ""

def read_headers(path, sheet_name=None):
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name] if sheet_name else wb.worksheets[0]
    headers = [ws.cell(row=1, column=c).value for c in range(1, ws.max_column + 1)]
    wb.close()
    return headers

def last_used_row_in_col(ws, col_idx: int) -> int:
    """Finde letzte befüllte Zeile in Spalte col_idx (>=1)."""
    col_letter = get_column_letter(col_idx)
    for cell in reversed(list(ws[col_letter])):
        if cell.value not in (None, ""):
            return cell.row
    return 1

def main():
    # 1) Maestroni-Header
    maestro_headers_raw = read_headers(MAESTRONI_PATH, MAESTRONI_SHEET)
    maestro_headers = {norm(h): h for h in maestro_headers_raw if norm(h)}

    # 2) Unsere Header
    unsere_headers_raw = read_headers(UNSERE_PATH, UNSERE_SHEET)
    unsere_norm_set = {norm(h) for h in unsere_headers_raw if norm(h)}

    # 3) Treffer-Spalten bestimmen
    hit_cols = [(c, hdr) for c, hdr in enumerate(unsere_headers_raw, start=1)
                if norm(hdr) in maestro_headers and norm(hdr)]

    # 4) Datei kopieren → nur in Kopie arbeiten
    base, ext = os.path.splitext(UNSERE_PATH)
    out_path = base + "_highlighted" + ext
    shutil.copyfile(UNSERE_PATH, out_path)

    # 5) Kopie öffnen und färben
    wb = load_workbook(out_path)
    ws = wb[UNSERE_SHEET] if UNSERE_SHEET else wb.worksheets[0]

    for c, hdr in hit_cols:
        last_row = last_used_row_in_col(ws, c)
        for r in range(1, last_row + 1):
            ws.cell(row=r, column=c).fill = YELLOW

    wb.save(out_path)
    wb.close()

    # 6) Reporting
    print(f"\nGefundene Übereinstimmungen: {len(hit_cols)}")
    for c, hdr in hit_cols:
        print(f"  Spalte {c}: {hdr}")

    fehlende = [orig for normed, orig in maestro_headers.items() if normed not in unsere_norm_set]
    print(f"\nIn Maestroni, aber NICHT in unserer Tabelle gefunden: {len(fehlende)}")
    for h in fehlende:
        print(f"  {h}")

    print(f"\nErgebnis gespeichert als: {out_path}")

if __name__ == "__main__":
    main()



Gefundene Übereinstimmungen: 20
  Spalte 1: Verletzungsstatus
  Spalte 2: Geschlecht_weiblich
  Spalte 3: INV_CMJ_uni_Rel. Peak Power 
  Spalte 4: UNINV_CMJ_uni_Rel. Peak Power 
  Spalte 5: INV_CMJ_uni_Peak braking force 
  Spalte 6: UNINV_CMJ_uni_Peak braking force 
  Spalte 7: INV_CMJ_uni_Av. propulsive force 
  Spalte 8: UNINV_CMJ_uni_Av. propulsive force 
  Spalte 9: CMJ_Propulsive duration
  Spalte 10: INV_CMJ_uni_Braking Duration-Mittelwert [s]
  Spalte 11: INV_CMJ_uni_Peak Braking Velocity-Mittelwert [m/s]
  Spalte 12: INV_CMJ_uni_Countermovement Tiefe-Mittelwert [cm]
  Spalte 13: INV_CMJ_uni_Relative Peak Landing Force-Mittelwert [BW]
  Spalte 14: UNINV_CMJ_uni_Braking Duration-Mittelwert [s]
  Spalte 15: UNINV_CMJ_uni_Peak Braking Velocity-Mittelwert [m/s]
  Spalte 16: UNINV_CMJ_uni_Countermovement Tiefe-Mittelwert [cm]
  Spalte 17: UNINV_CMJ_uni_Relative Peak Landing Force-Mittelwert [BW]
  Spalte 18: INV_ISO_Max Extension 
  Spalte 19: UNINV_ISO_Max Extension 
  Spalte 20: 

### Matching Variablen in Spalte 1 und 2 markieren

In [5]:
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# ==== Pfad/Sheet ====
xlsx_path = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Metrik_Matching\Fehlende_Metriken_finden.xlsx"
sheet_name = None  # None = erstes Blatt; oder z.B. "Tabelle1"

# ==== Farben (ARGB) ====
YELLOW = PatternFill(fill_type="solid", start_color="FFFFEB84", end_color="FFFFEB84")  # Match
BLUE   = PatternFill(fill_type="solid", start_color="FFB7E1FF", end_color="FFB7E1FF")  # No-Match

def norm(s) -> str:
    """Alle Whitespaces entfernen + lowercase; None -> ''."""
    if s is None:
        return ""
    s = str(s)
    s = re.sub(r"\s+", "", s)  # entfernt Leerzeichen, Tabs, Zeilenumbrüche
    return s.lower()

# ==== Workbook laden ====
wb = load_workbook(xlsx_path)
ws = wb[sheet_name] if sheet_name else wb.worksheets[0]

# ==== Spalte A & B lesen (ab Zeile 2) ====
rows = ws.max_row
colA_raw = [ws.cell(row=r, column=1).value for r in range(2, rows+1)]
colB_raw = [ws.cell(row=r, column=2).value for r in range(2, rows+1)]

# Normalisierte Sets für schnellen Look-up
setA = {norm(v) for v in colA_raw if norm(v) != ""}
setB = {norm(v) for v in colB_raw if norm(v) != ""}

# ==== Färben: A gegen B, B gegen A ====
for r in range(2, rows+1):
    cA = ws.cell(row=r, column=1)
    cB = ws.cell(row=r, column=2)

    a_norm = norm(cA.value)
    b_norm = norm(cB.value)

    # Spalte A: ist der (normierte) Wert irgendwo in Spalte B?
    if a_norm != "":
        if a_norm in setB:
            cA.fill = YELLOW
        else:
            cA.fill = BLUE

    # Spalte B: ist der (normierte) Wert irgendwo in Spalte A?
    if b_norm != "":
        if b_norm in setA:
            cB.fill = YELLOW
        else:
            cB.fill = BLUE

# Hinweis: Leere Zellen bleiben unverändert. Falls du sie blau willst:
# if a_norm == "": cA.fill = BLUE
# if b_norm == "": cB.fill = BLUE

# ==== Speichern ====
wb.save(xlsx_path)
print(f"Fertig: Matches gelb, Non-Matches blau in '{xlsx_path}'.")


Fertig: Matches gelb, Non-Matches blau in 'K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Metrik_Matching\Fehlende_Metriken_finden.xlsx'.


### Umsortierung unsere Basistabelle (Angleichung der Reihenfolge wie Maestroni für guten Vergleich)

In [4]:
# -*- coding: utf-8 -*-
import unicodedata
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from openpyxl import load_workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.utils import get_column_letter

FILE_PATH = r"K:\Team\Böhmer_Michael\PAPER\Validierungsdaten_INV_UNINV\Basistabelle_ML_reduziert.xlsx"
SHEET_NAME = "Basis"                      

# Umsortierbereich C–T (3–20), Block 3–111 (inkl. Ist-Header in Zeile 3)
COL_START = 3    # C
COL_END   = 20   # T
ROW_DATA_START = 3     # <-- Block beginnt in Zeile 3 (Ist-Header)
ROW_DATA_END   = 111   # <-- Daten bis Zeile 111

# Zielreihenfolge kommt aus Zeile 1 (C..T)
ROW_TARGET_ORDER = 1
NEW_SHEET_NAME   = "ACLR_reordered"

def normalize_label(s: Optional[str]) -> str:
    if s is None:
        return ""
    s = str(s).strip()
    if not s:
        return ""
    s = unicodedata.normalize("NFKD", s)
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    for ch in ["\n", "\r", "\t", "_", "-", "–", "—", "/", "\\", "|"]:
        s = s.replace(ch, " ")
    return " ".join(s.split()).lower()

def read_cell(ws: Worksheet, row: int, col: int):
    return ws.cell(row=row, column=col).value

def write_cell(ws: Worksheet, row: int, col: int, value):
    ws.cell(row=row, column=col, value=value)

def get_target_order(ws: Worksheet) -> Tuple[List[str], List[str]]:
    raw, norm = [], []
    for c in range(COL_START, COL_END + 1):
        v = read_cell(ws, ROW_TARGET_ORDER, c)
        raw.append(v if v is not None else "")
        norm.append(normalize_label(v))
    return raw, norm

def get_source_labels_from_row3(ws: Worksheet) -> Tuple[List[str], List[str]]:
    """Liest die Ist-Header aus Zeile 3 des Umsortierbereichs (C3..T3)."""
    raw, norm = [], []
    for c in range(COL_START, COL_END + 1):
        v = read_cell(ws, ROW_DATA_START, c)  # Zeile 3
        raw.append(v if v is not None else "")
        norm.append(normalize_label(v))
    return raw, norm

def build_mapping(src_norm: List[str], tgt_norm: List[str]):
    label_to_tgt = {}
    for i, lab in enumerate(tgt_norm):
        label_to_tgt.setdefault(lab, []).append(i)

    dup_targets = {lab: idxs for lab, idxs in label_to_tgt.items() if len(idxs) > 1 and lab != ""}

    colmap: Dict[int, int] = {}
    used_tgt = set()
    for s_idx, slab in enumerate(src_norm):
        if slab == "":
            continue
        for ti in label_to_tgt.get(slab, []):
            if ti not in used_tgt:
                colmap[s_idx] = ti
                used_tgt.add(ti)
                break

    missing_targets = [ti for ti in range(len(tgt_norm)) if ti not in used_tgt]
    extras_sources  = [si for si in range(len(src_norm)) if si not in colmap]
    return colmap, missing_targets, extras_sources, dup_targets

def buffer_blocks(ws: Worksheet, colmap: Dict[int, int]) -> Dict[int, List]:
    """Puffert den gesamten Block Zeile 3–111 der jeweils zugeordneten Quellspalte."""
    buf: Dict[int, List] = {}
    for s_idx, t_idx in colmap.items():
        abs_col = COL_START + s_idx
        data = [read_cell(ws, r, abs_col) for r in range(ROW_DATA_START, ROW_DATA_END + 1)]
        buf[t_idx] = data
    return buf

def create_or_replace_sheet(wb, name: str) -> Worksheet:
    if name in wb.sheetnames:
        wb.remove(wb[name])
    return wb.create_sheet(title=name)

def main():
    xlsx = Path(FILE_PATH)
    if not xlsx.exists():
        raise FileNotFoundError(f"File not found: {xlsx}")
    wb = load_workbook(xlsx)
    if SHEET_NAME not in wb.sheetnames:
        raise ValueError(f"Sheet '{SHEET_NAME}' not found.")
    ws_src: Worksheet = wb[SHEET_NAME]

    # Ziel (Zeile 1) und Ist (Zeile 3) lesen
    tgt_raw, tgt_norm = get_target_order(ws_src)
    src_raw, src_norm = get_source_labels_from_row3(ws_src)

    colmap, missing_targets, extras_sources, dup_targets = build_mapping(src_norm, tgt_norm)

    # Phase 1: Puffer
    buffer = buffer_blocks(ws_src, colmap)

    # Phase 2: Schreiben in neues Blatt
    ws_out = create_or_replace_sheet(wb, NEW_SHEET_NAME)

    # (Optional) Linke Spalten A..B kopieren (Zeilen 1..ROW_DATA_END)
    for c in range(1, COL_START):
        for r in range(1, ROW_DATA_END + 1):
            write_cell(ws_out, r, c, read_cell(ws_src, r, c))

    # Zeile 1 (Ziel-Header) setzen
    for i, raw_label in enumerate(tgt_raw):
        write_cell(ws_out, ROW_TARGET_ORDER, COL_START + i, raw_label)

    # Zeile 2 bewusst leer lassen
    for i in range(COL_END - COL_START + 1):
        write_cell(ws_out, 2, COL_START + i, None)

    # Block Zeilen 3–111 gemäß Mapping schreiben
    for t_idx in range(COL_END - COL_START + 1):
        abs_col_out = COL_START + t_idx
        if t_idx in buffer:
            data = buffer[t_idx]
            for off, val in enumerate(data):
                write_cell(ws_out, ROW_DATA_START + off, abs_col_out, val)
        else:
            # fehlende Zielspalte leer lassen
            for r in range(ROW_DATA_START, ROW_DATA_END + 1):
                write_cell(ws_out, r, abs_col_out, None)

    # Extras rechts anhängen (optional)
    if extras_sources:
        out_c = COL_END + 1
        for s_idx in extras_sources:
            abs_col_src = COL_START + s_idx
            # kopiere auch Zeilen 1..ROW_DATA_END für Transparenz
            for r in range(1, ROW_DATA_END + 1):
                write_cell(ws_out, r, out_c, read_cell(ws_src, r, abs_col_src))
            out_c += 1

    # Report
    print("=== Reorder Summary ===")
    print(f"Source sheet  : {SHEET_NAME}")
    print(f"Output sheet  : {NEW_SHEET_NAME}")
    print(f"Block moved   : {get_column_letter(COL_START)}{ROW_DATA_START}–{get_column_letter(COL_END)}{ROW_DATA_END}")
    print(f"Mapped columns: {len(colmap)} / {COL_END - COL_START + 1}")
    if dup_targets:
        print("Duplicate target labels (normalized):")
        for lab, idxs in dup_targets.items():
            cols = [get_column_letter(COL_START + i) for i in idxs]
            print(f"  '{lab}' at targets {cols}")
    if missing_targets:
        cols = [get_column_letter(COL_START + i) for i in missing_targets]
        names = [tgt_raw[i] for i in missing_targets]
        print(f"Missing targets: {list(zip(cols, names))}")
    if extras_sources:
        cols = [get_column_letter(COL_START + i) for i in extras_sources]
        names = [src_raw[i] for i in extras_sources]
        print(f"Extra sources : {list(zip(cols, names))}")

    wb.save(xlsx)
    print(f"Saved: {xlsx}")
    print("Done.")

if __name__ == "__main__":
    main()


=== Reorder Summary ===
Source sheet  : Basis
Output sheet  : ACLR_reordered
Block moved   : C3–T111
Mapped columns: 18 / 18
Saved: K:\Team\Böhmer_Michael\PAPER\Validierungsdaten_INV_UNINV\Basistabelle_ML_reduziert.xlsx
Done.


### Frauen exkludieren

In [1]:
import pandas as pd

# Datei-Pfad
file_path = r"K:\Team\Böhmer_Michael\PAPER\Finale_Versionen_ML_Vergleich\Basistabelle_ML_vkb_male.xlsx"

# Excel einlesen
df = pd.read_excel(file_path, engine="openpyxl")

# Filtern: nur Zeilen behalten, wo Spalte 2 != 1
# Spalte 2 = df.iloc[:,1]
df_filtered = df[df.iloc[:, 1] != 1]

# Gefiltertes DataFrame zurückschreiben (überschreibt Originaldatei)
df_filtered.to_excel(file_path, index=False, engine="openpyxl")

print("Fertig. Zeilen mit '1' in Spalte 2 wurden entfernt.")


Fertig. Zeilen mit '1' in Spalte 2 wurden entfernt.


### Gewichtsnormalisieren

In [ ]:
import pandas as pd
from pathlib import Path

# ====== Konfiguration ======
input_path  = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Feature_Selektion_Maestroni_Daten\Basistabelle_ML_Maestroni_gewichtsnormalisiert - Kopie.xlsx"
sheet_name  = 0             # erstes Blatt; alternativ Blattname als String
weight_col_excel = 2        # 1-basiert (Excel-Logik): Spalte 2 = Gewicht
cols_to_norm_excel = [6, 9, 10, 14, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25,
                      29, 30, 31, 32, 35, 36, 37, 39, 40, 44, 45, 47, 50, 51,
                      52, 53, 56, 57, 58, 60, 61, 65, 66, 68]

# Ausgabe: None => überschreibt die Eingabedatei
output_path = None  # oder z.B.: r"K:\...\Basistabelle_ML_Maestroni_normiert.xlsx"

# ====== Logik ======
# Excel (1-basiert) -> pandas (0-basiert) Indizes
weight_idx = weight_col_excel - 1
cols_to_norm_idx = [c - 1 for c in cols_to_norm_excel]

# Datei laden (erste Zeile = Header, Daten ab Zeile 2)
df = pd.read_excel(input_path, sheet_name=sheet_name)

# Gewicht-Spalte extrahieren und in numerisch umwandeln
weight = pd.to_numeric(df.iloc[:, weight_idx], errors="coerce")

# Warnung bei fehlerhaften Gewichten
n_zero = (weight == 0).sum(skipna=True)
n_neg  = (weight < 0).sum(skipna=True)
n_nan  = weight.isna().sum()
if n_zero or n_neg or n_nan:
    #print(f"[Hinweis] Gewicht: {n_zero}×0, {n_neg}×negativ, {n_nan}×NaN – betroffene Zeilen ergeben NaN nach Division.")

# Normalisierung (überschreiben direkt in df)
for idx in cols_to_norm_idx:
    col_vals = pd.to_numeric(df.iloc[:, idx], errors="coerce")
    df.iloc[:, idx] = col_vals / weight

# Speichern
if output_path is None:
    save_path = input_path
else:
    save_path = output_path

df.to_excel(save_path, index=False)
print(f"Gewichtsnormalisierung abgeschlossen. Gespeichert unter:\n{save_path}")


c:\Users\boehmer\neuronales_netz_motum\.venv\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
c:\Users\boehmer\neuronales_netz_motum\.venv\lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Conditional Formatting extension is not supported and will be removed
  warn(msg)
C:\Users\boehmer\AppData\Local\Temp\ipykernel_24672\2153637094.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     29.563830
1     30.656716
2     23.312500
3     26.800000
4     28.537005
        ...    
70    35.277008
71    27.405542
72    28.333333
73    26.607370
74    29.283582
Length: 75, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.iloc[:, idx] = col_vals / weight
C:\Users\boehmer\AppData\Local\Temp\ipykernel_24672\2153637094.py:36: FutureWarning: Setting an item of incompat

Gewichtsnormalisierung abgeschlossen. Gespeichert unter:
K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Feature_Selektion_Maestroni_Daten\Basistabelle_ML_Maestroni_gewichtsnormalisiert - Kopie.xlsx


In [2]:
import pandas as pd
import warnings

# ====== alle Pandas FutureWarnings unterdrücken ======
warnings.filterwarnings("ignore", category=FutureWarning)

# ====== Konfiguration ======
input_path  = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Feature_Selektion_Maestroni_Daten\Basistabelle_ML_Maestroni_gewichtsnormalisiert - Kopie.xlsx"
sheet_name  = 0
weight_col_excel = 2

# neue Spalten, die nur in Zeile 2–41 normalisiert werden sollen
cols_partial_norm_excel = [70, 71, 73, 74]

# Bereich: Zeilen 2–41 (Excel-Logik) -> Pandas Indizes 0–39
row_start = 0   # Zeile 2 in Excel = Index 0
row_end   = 39  # Zeile 41 in Excel = Index 39

# ====== Logik ======
# Excel (1-basiert) -> pandas (0-basiert)
weight_idx = weight_col_excel - 1
cols_partial_idx = [c - 1 for c in cols_partial_norm_excel]

# Datei laden
df = pd.read_excel(input_path, sheet_name=sheet_name)

# Gewicht-Spalte (Zeilen 2–41)
weight = pd.to_numeric(df.iloc[row_start:row_end+1, weight_idx], errors="coerce")

# Teilnormalisierung durchführen
for idx in cols_partial_idx:
    col_vals = pd.to_numeric(df.iloc[row_start:row_end+1, idx], errors="coerce")
    df.iloc[row_start:row_end+1, idx] = col_vals / weight

# Datei speichern (überschreibt Original)
df.to_excel(input_path, index=False)
print(f"Teil-Gewichtsnormalisierung abgeschlossen (Zeilen 2–41, Spalten {cols_partial_norm_excel}).")


Teil-Gewichtsnormalisierung abgeschlossen (Zeilen 2–41, Spalten [70, 71, 73, 74]).


### Metriken klassische ML-Modelle

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, BaggingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, recall_score
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)

def repeated_k_fold(model, X, y, n_splits=5, n_repeats=10):
    rkf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=42)

    accuracy_train, accuracy_test = [], []
    f1, recall, roc_auc = [], [], []

    for train_index, test_index in rkf.split(X, y):
        # Aufteilen in Trainings- und Testdaten
        X_train, X_test = X[train_index], X[test_index]
        y_train, y_test = y[train_index], y[test_index]

        # Skalierung: nur an den Trainingsdaten fitten
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        model.fit(X_train_scaled, y_train)

        y_train_pred = model.predict(X_train_scaled)
        y_test_pred = model.predict(X_test_scaled)

        accuracy_train.append(accuracy_score(y_train, y_train_pred))
        accuracy_test.append(accuracy_score(y_test, y_test_pred))
        f1.append(f1_score(y_test, y_test_pred))
        recall.append(recall_score(y_test, y_test_pred))
        roc_auc.append(roc_auc_score(y_test, model.predict_proba(X_test_scaled)[:, 1]))

    return {
        "Train Accuracy": (np.mean(accuracy_train), np.std(accuracy_train)),
        "Test Accuracy": (np.mean(accuracy_test), np.std(accuracy_test)),
        "F1-Score_1": (np.mean(f1), np.std(f1)),
        "Recall_1": (np.mean(recall), np.std(recall)),
        "ROC-AUC_1": (np.mean(roc_auc), np.std(roc_auc)),
    }


# Pfad zur Datei
file_path = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Basistabelle_Maestroni_ML.xlsx"

try: 
    df = pd.read_excel(file_path)
    
    # Zielvariable (y) und Features (X) extrahieren
    y = df['Verletzungsstatus']
    
    # Dummy-Variable "Geschlecht_weiblich" separieren, falls vorhanden
    if 'Geschlecht_weiblich' in df.columns:
        geschlecht_weiblich = df[['Geschlecht_weiblich']]
        X = df.drop(columns=['Verletzungsstatus', 'Geschlecht_weiblich'])
        # Die Dummy-Variable wieder anhängen
        X = np.hstack((X.values, geschlecht_weiblich.values))
    else:
        X = df.drop(columns=['Verletzungsstatus']).values
    
    # Modelle definieren
    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "Random Forest": RandomForestClassifier(random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
        "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42),
        "LightGBM": LGBMClassifier(verbose=-1, random_state=42),
        "SVC": SVC(probability=True, random_state=42),
        "k-Nearest Neighbors": KNeighborsClassifier(),
        "MLP Classifier": MLPClassifier(max_iter=1000, random_state=42),
        "Gaussian Naive Bayes": GaussianNB(),
        "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
        "Quadratic Discriminant Analysis": QuadraticDiscriminantAnalysis(),
        "Bagging Classifier": BaggingClassifier(random_state=42),
        "Extra Trees": ExtraTreesClassifier(random_state=42),
    }
    
    results = []
    for model_name, model in models.items():
        print(f"Modell wird validiert: {model_name}")
        metrics = repeated_k_fold(model, X, y)
        
        formatted_metrics = {
            "Model": model_name,
            "Train Accuracy": f"{metrics['Train Accuracy'][0]:.4f} ± {metrics['Train Accuracy'][1]:.4f}",
            "Test Accuracy": f"{metrics['Test Accuracy'][0]:.4f} ± {metrics['Test Accuracy'][1]:.4f}",
            "F1-Score_1": f"{metrics['F1-Score_1'][0]:.4f} ± {metrics['F1-Score_1'][1]:.4f}",
            "Recall_1": f"{metrics['Recall_1'][0]:.4f} ± {metrics['Recall_1'][1]:.4f}",
            "ROC-AUC_1": f"{metrics['ROC-AUC_1'][0]:.4f} ± {metrics['ROC-AUC_1'][1]:.4f}",
        }
        results.append(formatted_metrics)
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by="ROC-AUC_1", ascending=False)
    
    print("\nErgebnisse der Modelle:")
    print(results_df)


except FileNotFoundError:
    print("Die Datei wurde nicht gefunden. Bitte überprüfe den Pfad.")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")

### Finale Metriken sortieren

In [2]:
import pandas as pd

# Pfade
in_path = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Metrik_Matching\Matched_Maestroni_Unsere_Daten.xlsx"
out_path = r"K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Metrik_Matching\Matched_Maestroni_Unsere_Daten_sorted.xlsx"

# Einlesen
df = pd.read_excel(in_path)

# Zeilen 28–33 (1-basiert) fixieren
fixed = df.iloc[27:33]   # Pandas ist 0-basiert
others_top = df.iloc[:27]
others_bottom = df.iloc[33:]

# Nur die „beweglichen“ Teile zusammenfassen
others = pd.concat([others_top, others_bottom])

# Sortierlogik
def sort_key(val):
    v = str(val)
    if v.startswith("INV"):
        return (2, v)  # nach den Einzelmetriken, aber vor UNINV
    elif v.startswith("UNINV"):
        return (3, v)
    else:
        return (1, v)  # Einzelmetriken zuerst

# Sortieren nach Spalte 1, bei Gleichheit auch nach Spalte 2
others_sorted = others.sort_values(
    by=[df.columns[0], df.columns[1]],
    key=lambda col: col.map(sort_key)
)

# Wieder zusammensetzen: sortierte Teile + fixe Zeilen
final_df = pd.concat([others_sorted.iloc[:27], fixed, others_sorted.iloc[27:]], ignore_index=True)

# Speichern
final_df.to_excel(out_path, index=False)

print(f"Fertig! Datei gespeichert unter:\n{out_path}")


Fertig! Datei gespeichert unter:
K:\Team\Böhmer_Michael\PAPER\Neuer_Ansatz\Metrik_Matching\Matched_Maestroni_Unsere_Daten_sorted.xlsx


### Finale Dateisets bauen